In [ ]:
%config InlineBackend.figure_formats = ['svg']
from IPython.display import display, HTML
display(HTML("<style>body { max-width: 1200px; margin: auto; }</style>"))

import warnings
warnings.simplefilter('ignore')
from pathlib import Path
import json
root_dir = Path('../../data/annotation/')
manual_dir = root_dir / 'annotated'


def load_ndjson(filename):
    with open(filename) as f:
        return [json.loads(l) for l in f]
names = 'ap_whole', 'lateral_whole'
jsons = {n:[] for n in names}
for name in names:
    for fn in manual_dir.glob(f'**/valid_{name}.ndjson'):
        # print(fn)
        ndjson = load_ndjson(fn)
        for j in ndjson:
            j['ndjson_dir'] = str(fn.parent.name)
        with open(fn) as f:
            for line in f:
                data = json.loads(line)
                data['ndjson_dir'] = str(fn.parent.name)
                jsons[name].append(data)
    print('The number of annotation in', name, ':', len(jsons[name]))

In [ ]:
import pandas as pd
rows = []
for name, lines in jsons.items():
    for line in lines:
        rows.append((name, line['ndjson_dir'], line['filepath']))
df = pd.DataFrame(rows, columns=['ScanType','CollectionDate','filepath'])

In [ ]:
df['filename'] = df.filepath.map(lambda p:Path(p).name)
df['PatientID'] = df.filepath.map(lambda p:Path(p).name.split('_')[0])
df

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["svg.fonttype"] = 'none'
plt.rcParams['font.family'] = 'serif'
import seaborn as sns
sns.set(context='notebook', style='whitegrid', font='sans-serif')
df_count = pd.DataFrame(df.groupby(['ScanType','CollectionDate'])['PatientID'].count()).reset_index()
df_count['累計'] = df_count.groupby('ScanType').cumsum()
ax = sns.barplot(data=df_count, x='ScanType', y='累計', hue='CollectionDate')
for c in ax.containers:
    ax.bar_label(c)
plt.title('画像枚数')

In [ ]:
df_count = pd.DataFrame(df.groupby(['ScanType','CollectionDate'])['PatientID'].nunique()).reset_index()
df_count['累計'] = df_count.groupby('ScanType').cumsum()
ax = sns.barplot(data=df_count, x='ScanType', y='累計', hue='CollectionDate')
for c in ax.containers:
    ax.bar_label(c)
plt.title('症例数')

In [ ]:
import sys
sys.path.append('../szkspyne')
import labelme_utils


import numpy as np
from scipy.spatial.distance import cdist
POINT_NAMES = ['TL', 'TR', 'BL', 'BR']
from PIL import Image
import matplotlib.pyplot as plt
def ticks_off():
    plt.xticks([])
    plt.yticks([])
    [s.set_visible(False) for s in plt.gca().spines.values()]

def json2imgpath(filename):
    image_filename = (img_dir / '/'.join(filename.split('_'))).with_suffix('.jpg')
    return image_filename

img_dir = Path('../../data/jpg_half') 
dates = ['20220524','20220606','20220628']
for direction in ['ap_whole', 'lateral_whole']:
    display(HTML(f'<h2>{direction}</h2>'))
    data_per_date = {}
    for date in dates:
        jsons = load_ndjson(root_dir / date / f'AI_{direction}.jsonl')
        data_per_date[date] = {j['filename']:labelme_utils.conv2shape_dict(j) for j in jsons}

    dfs = []
    for date_prev, date_cur in zip(dates, dates[1:]):
        prev, cur = data_per_date[date_prev], data_per_date[date_cur]
        mean_cur2prev = {}
        for filename, data_cur in cur.items():
            if filename not in prev:
                continue
            d_data_prev = prev[filename]
            d_data_cur = cur[filename]
            mins = []
            for POINT in POINT_NAMES:
                pts_prev = np.array(d_data_prev['point'][POINT]).squeeze()
                pts_cur = np.array(d_data_cur['point'][POINT]).squeeze()
                if pts_prev.ndim!=2 or pts_cur.ndim!=2:
                    continue
                dists = cdist(pts_prev, pts_cur)
                min_cur2prev = dists.min(axis=0)
                mins.append(min_cur2prev)
            if mins:
                mean_cur2prev[filename] = (np.concatenate(mins).mean())
        df_dist = pd.DataFrame(pd.Series(mean_cur2prev), columns=['MeanDist'])
        df_dist.sort_values('MeanDist', inplace=True)
        df_dist['dates'] = f'{date_prev} vs. {date_cur}'
        df_dist.index.name = 'filename'
        dfs.append(df_dist.reset_index())
    df_dist = pd.concat(dfs, axis=0, ignore_index=True)
    sns.histplot(df_dist, x='MeanDist', hue='dates')
    plt.show()

    min_filenames = [df_dist.iloc[i].filename for i in [0, len(df_dist)//2, -1]]
    min_names= ['min','median','max']
    plt.figure(figsize=(8, 5))
    MARGIN = 50
    for i, (name, min_filename) in enumerate(zip(min_names, min_filenames)):
        plt.subplot(1, len(min_filenames), i+1)
        img = Image.open(json2imgpath(min_filename))
        plt.imshow(img, 'gray')
        for d, label, color in [(prev[min_filename], date_prev, 'r'), (cur[min_filename], date_cur, 'b')]:
            pts = []
            for POINT in POINT_NAMES:
                pts.append(np.array(d['point'][POINT]).squeeze())
            try:
                pts = np.concatenate(pts)
                plt.scatter(pts[:, 0], pts[:, 1], label=label, s=8, linewidths=1, alpha=.7, c=color, edgecolors=color)
            except:
                pass
        bbox = np.array((pts.min(axis=0)-MARGIN, pts.max(axis=0)+MARGIN))
        plt.xlim(bbox[:, 0][::-1])
        plt.ylim(bbox[:, 1][::-1])
        plt.legend()
        plt.xlabel(name)
        ticks_off()
    plt.show()